In [1]:
# 1. Montar Google Drive (seguro e idempotente)
from google.colab import drive
import os, subprocess, warnings
warnings.filterwarnings("ignore")

MNT = "/content/drive"
if not os.path.ismount(MNT):
    if os.path.exists(MNT) and os.listdir(MNT):
        try:
            subprocess.run(["fusermount", "-u", MNT], check=False)
        except Exception:
            pass
        !rm -rf /content/drive/*
    drive.mount(MNT, force_remount=True)

ROOT = "MyDrive" if os.path.exists("/content/drive/MyDrive") else "MeuDrive"
BASE_PATH = f"/content/drive/{ROOT}/TCC_USP"
PROC_PATH = os.path.join(BASE_PATH, "data_processed")

print("PROC_PATH:", PROC_PATH)

Mounted at /content/drive
PROC_PATH: /content/drive/MyDrive/TCC_USP/data_processed


In [2]:
# 2. Carregar dados
import pandas as pd, numpy as np

ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)
if "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

if "clean_text" not in df_news.columns:
    df_news["clean_text"] = df_news.get("texto","").fillna("")

print("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)

IBOV: (20, 2) | NEWS: (81, 6)


In [3]:
# 3. Agregar textos por dia e criar target do IBOV
df_text = df_news.groupby("data", as_index=False)["clean_text"].apply(
    lambda s: " ".join([str(x) for x in s if str(x).strip()])
)

df_ibov = df_ibov.sort_values("data").reset_index(drop=True)
df_ibov["ret1"] = df_ibov["close"].pct_change().shift(-1)
df_ibov["y"] = (df_ibov["ret1"] > 0).astype(int)
df_target = df_ibov.dropna(subset=["ret1"]).copy()

df = pd.merge(df_target[["data","close","y"]], df_text, on="data", how="inner").sort_values("data")

# fallback dummy
if df.shape[0] == 0:
    print("⚠️ Nenhuma interseção → usando dataset dummy.")
    df = pd.DataFrame({
        "data": pd.date_range("2024-01-02", periods=8, freq="D"),
        "close": np.linspace(130000,130700,8),
        "y": [0,1,0,1,1,0,1,0],
        "clean_text": [
            "mercado alta otimismo",
            "queda dolar pressao",
            "petrobras avanco dividendos",
            "ibovespa estabilidade noticias",
            "crescimento economia brasil",
            "inflacao preocupa investidores",
            "mercado futuro reage positivo",
            "politica monetaria em foco"
        ]
    })

print("Após merge/dummy:", df.shape)
display(df.head())

⚠️ Nenhuma interseção → usando dataset dummy.
Após merge/dummy: (8, 4)


,data,close,y,clean_text
0,2024-01-02,130000.0,0,mercado alta otimismo
1,2024-01-03,130100.0,1,queda dolar pressao
2,2024-01-04,130200.0,0,petrobras avanco dividendos
3,2024-01-05,130300.0,1,ibovespa estabilidade noticias
4,2024-01-06,130400.0,1,crescimento economia brasil


In [4]:
# 4. Embeddings (Sentence Transformers)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer("all-MiniLM-L6-v2")

sentences = df["clean_text"].fillna("").replace("", "vazio").tolist()
X = model_emb.encode(sentences, convert_to_numpy=True, show_progress_bar=True)
y = df["y"].astype(int).reset_index(drop=True)

print("Matriz Embeddings:", X.shape, "| y:", y.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Matriz Embeddings: (8, 384) | y: (8,)


In [5]:
# 5. Função de avaliação (mesma lógica do TF-IDF)
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

def evaluate_timeaware(X, y, models):
    n = X.shape[0]
    results = {}
    if n < 2:
        print(f"⚠️ Dataset com {n} amostra(s). Impossível treinar.")
        return results

    if n >= 30: n_splits = 5
    elif n >= 10: n_splits = 3
    elif n >= 5: n_splits = 2
    else: n_splits = 0

    if n_splits >= 2:
        tscv = TimeSeriesSplit(n_splits=n_splits)
        for name, model in models.items():
            aucs, mdas = [], []
            for tr, te in tscv.split(np.arange(n)):
                model.fit(X[tr], y.iloc[tr])
                proba = model.predict_proba(X[te])[:,1]
                pred  = (proba > 0.5).astype(int)
                aucs.append(roc_auc_score(y.iloc[te], proba))
                mdas.append(accuracy_score(y.iloc[te], pred))
            results[name] = {"AUC": float(np.mean(aucs)), "MDA": float(np.mean(mdas))}
        print(f"✅ Avaliação com TimeSeriesSplit (n_splits={n_splits}, n={n})")
    else:
        print(f"⚠️ Poucas amostras (n={n}). Usando holdout 50/50.")
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, shuffle=False)
        for name, model in models.items():
            model.fit(X_train, y_train)
            proba = model.predict_proba(X_test)[:,1]
            pred  = (proba > 0.5).astype(int)
            results[name] = {
                "AUC": float(roc_auc_score(y_test, proba)) if len(np.unique(y_test)) > 1 else float("nan"),
                "MDA": float(accuracy_score(y_test, pred))
            }
    return results

In [6]:
# 6. Rodar modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42)
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
    )
except:
    print("ℹ️ XGBoost não disponível.")

results = evaluate_timeaware(X, y, models)
print("\nResultados médios (Embeddings real/dummy):")
print(results)

✅ Avaliação com TimeSeriesSplit (n_splits=2, n=8)

Resultados médios (Embeddings real/dummy):
{'Logistic': {'AUC': 0.0, 'MDA': 0.25}, 'RandomForest': {'AUC': 1.0, 'MDA': 0.5}, 'XGBoost': {'AUC': 0.5, 'MDA': 0.5}}
